# Embeddings generation with Ollama

Turns the PDF in `data/` into chunks, embeds them with **`nomic-embed-text`** through Ollama, and saves a FAISS vector store to `vectorstores/supplements_ollama/` together with a `manifest.json` (embedding model, dimension, file hashes).

**Prerequisites** (see the README for details)

1. Install [Ollama](https://ollama.com) and make sure it is running (`ollama serve`, or the background service on Windows/macOS).
2. Download the model(s) used here:

```powershell
ollama pull nomic-embed-text
```
3. `pip install -r requirements.txt` inside a Python 3.12 virtual environment, and select that environment as this notebook's kernel.

Run cells top to bottom. *Author: Ratnesh Pathak*

> This notebook originally only *loaded* a pre-built `health_supplements` index (whose source PDF was not in the project) and failed with `name 'chunks' is not defined` when asked to build one. It now performs the whole build, so the index can be regenerated from the files in `data/`.

> **About the saved outputs:** this notebook was *not executed* during the repository audit (no Ollama server available), so no outputs are saved (no Ollama server was available). Model replies vary from run to run.

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable whether the notebook is started from the repo root or a sub-folder.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "rag_chatbot").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from rag_chatbot import config
from rag_chatbot.documents import load_documents, split_documents
from rag_chatbot.indexing import build_index, load_index

config.load_env()
EMBED_MODEL = config.get_ollama_embed_model()      # default: nomic-embed-text
INDEX_DIR = config.VECTORSTORES_DIR / "supplements_ollama"
config.check_ollama([EMBED_MODEL])                  # clear error if Ollama is down or the model is missing
print("Ollama OK, embedding model:", EMBED_MODEL)

### 1 · Load and split the document

In [ ]:
SOURCE = config.DEFAULT_PDF
docs = load_documents([SOURCE])                      # one Document per page, with source + page metadata
chunks = split_documents(docs, chunk_size=1000, chunk_overlap=150)
print(f"{len(docs)} pages -> {len(chunks)} chunks")
print(chunks[0].metadata)
print(chunks[0].page_content[:300])

### 2 · Embed and save the FAISS index

In [ ]:
embeddings = config.make_ollama_embeddings(EMBED_MODEL)
print("embedding dimension:", len(embeddings.embed_query("dimension probe")))   # 768 is expected for nomic-embed-text

manifest = build_index(chunks, embeddings, INDEX_DIR, provider="ollama", embedding_model=EMBED_MODEL,
                       source_files=[SOURCE], chunk_size=1000, chunk_overlap=150)
manifest

### 3 · How retrieval works

The question is embedded with the **same** model and compared with every chunk vector (FAISS, L2 distance). `load_index` refuses to load a store if its manifest/hashes are missing or the embedding model differs.

In [ ]:
vectorstore = load_index(INDEX_DIR, embeddings, expected_embedding_model=EMBED_MODEL)

question = "What does the review say about vitamin D supplementation?"
for doc, score in vectorstore.similarity_search_with_score(question, k=3):
    print(f"distance={score:.3f}  {doc.metadata['source']} p.{doc.metadata['page']}")
    print(doc.page_content[:200].replace("\n", " "), "\n")

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retriever.invoke(question)

**Troubleshooting:** `Cannot reach Ollama` -> start Ollama. `models are not installed` -> run the printed `ollama pull` command. Dimension mismatch / `Index was built with embedding model ...` -> rebuild the index with the model you are querying with.